# Models exploration

## Hierachical approach
- Since we know the taxonomy, a 2 level hierachical approach can be applied
- Start from training models on clean audio sound

### 1. Top-down approach (hard hierarchical combination)
- Model 1 : predict family
- A. Model 1...n_families : predict species conditioned by the family OR B. Single global model with masking.

#### Inference. Hard combination
During inference, predict family, then predict species.

**Problem** : wrong family -> wrong species (error propagation)

#### Inference. Soft combination
If model outputs probabilities, then
$$
P(y|x) = P(family_y | x) \cdot P(y | x, family_y)
$$

Also, instead of training n_families models, one model can be trained to predict species, then (with hypothesis that model implicitely learned hierachy),

$$
P(y|x, family_y) = P(y|x)
$$

## Train/test separation
Simple stratification per labels

## Features
#### Features extraction 
0. Waveform? 
1. MFCC (with delta, delta-delta)
2. Spectral features : useful since different species often occupy different frequency ranges.
- Spectral centroid -> "brightness"
- Spectral bandwidth -> spread of frequencies
- Spectral rolloff -> frequency below which X% energy lies
- Spectral contrast -> peaks vs valleys

3. RMS energy, zero-crossing rate (distinguish between tonal calls vs noisy env)

#### Pooling/aggregation
Since every audio is of different duration, extracted features must be aggregated.

##### Global pooling : 1 audio = 1 unit, aggregate over all frames
- Mean, std, max, min, percentiles, skewness & kurtosis

##### Chunking : 1 audio = N units/chunks, aggregate over chunks
- Split every audio into N chunks
- Aggregate per segment
- Concatenate aggregates features

## Questions
1. Training on clean audio, then how to adapt models to soundscape
2. Some classes are present only in soundscape, how to adapt models and detect these classes

In [1]:
import numpy as np

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from sklearn.preprocessing import MultiLabelBinarizer

import librosa
import librosa.display
import soundfile as sf

from IPython.display import Markdown

from birdclef_2026_ml.notebook_utils import init_notebook
from birdclef_2026_ml.paths import PATHS
from birdclef_2026_ml.audio_utils import load_train_audio
from birdclef_2026_ml.eda import *
from birdclef_2026_ml.preprocess import *
from birdclef_2026_ml.constants import SAMPLE_RATE


init_notebook()

In [8]:
train = pd.read_parquet(PATHS["clean_train"])

y, audio_path = load_train_audio(train, 0)
mfcc = librosa.feature.mfcc(y=y, sr=SAMPLE_RATE, n_mfcc=13)
np.concatenate([np.mean(mfcc, axis=1), np.std(mfcc, axis=1)])

array([-343.19025  ,   26.22139  ,   20.489183 ,   64.18163  ,
        -28.644892 ,   14.951253 ,   10.959044 ,    5.2486525,
        -15.337441 ,   19.631586 ,   -7.95491  ,    3.0573626,
          2.8470886,   47.45508  ,   50.4532   ,   15.823468 ,
         12.146823 ,   14.276966 ,    7.6806946,    8.0225725,
          6.623905 ,    8.798664 ,    7.565757 ,    5.6072817,
          5.120953 ,    5.232577 ], dtype=float32)